# North American Ecommerce Revenue Dashboard

This notebook reads the dbt-built DuckDB marts and shows the governed revenue story for Finance, Marketing, and Ecommerce.

**GitHub note:** Plotly charts are interactive when run locally. GitHub does not execute the Plotly JavaScript in notebook previews, so each chart includes a static SVG fallback exported from the same Plotly figure.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd
import plotly.express as px

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent

db_path = root / 'data' / 'ecommerce.duckdb'
con = duckdb.connect(str(db_path), read_only=True)

kpis = con.execute('select * from main.mart_revenue_kpis').df()
reconciliation = con.execute('select * from main.mart_revenue_reconciliation').df()
finance = con.execute('select * from main.mart_finance_monthly_revenue').df()
marketing = con.execute('select * from main.mart_marketing_attribution').df()
channel = con.execute('select * from main.mart_ecommerce_channel_performance').df()
quality = con.execute('select * from main.mart_data_quality_issues').df()
con.close()

## Executive KPIs

The KPI mart summarizes the size of the raw-to-governed revenue gap and the final governed reporting numbers.

In [2]:
kpis.T.rename(columns={0: 'value'})

,value
governed_orders,60.00
raw_order_rows,66.00
governed_gross_revenue,11344.44
returns_deducted,-794.96
governed_net_revenue,10549.48
promo_attributed_net_revenue,2749.86
promo_attributed_orders,18.00
cancelled_orders,4.00
returned_orders,4.00
raw_gross_revenue,13144.35


## Revenue Reconciliation

This bridge explains how raw gross revenue becomes governed net revenue.

In [3]:
reconciliation

,step_order,reconciliation_step,order_delta,revenue_adjustment,running_revenue,root_cause_layer
0,1,Raw extracted orders,66,13144.35,13144.35,Raw source
1,2,Less: duplicate retry loads removed,-3,-729.97,12414.38,Source and capture
2,3,Less: incomplete pipeline records excluded,-3,-259.98,12154.40,Pipeline and consumption
3,4,Less: cancelled order revenue zeroed,0,-809.96,11344.44,Semantic policy
4,5,Less: returns deducted and sign issues corrected,0,-794.96,10549.48,Semantic policy
5,6,Governed net revenue,<NA>,10549.48,10549.48,Governed fact


In [4]:
fig = px.bar(
    reconciliation[reconciliation['step_order'] < 6],
    x='reconciliation_step',
    y='revenue_adjustment',
    color='root_cause_layer',
    title='Raw-to-Governed Revenue Bridge',
    labels={'revenue_adjustment': 'Revenue Adjustment', 'reconciliation_step': 'Step'},
)
fig.update_layout(xaxis_tickangle=-25)
fig.show()

#### GitHub-renderable static fallback

This static image is exported from the Plotly chart for GitHub preview; the local notebook version remains interactive.

![Raw-to-governed revenue bridge](../assets/charts/revenue-reconciliation-bridge.svg)


## Finance View

Finance uses recognized net revenue by `payment_date`.

In [5]:
finance

,revenue_month,orders,gross_revenue,returns_deducted,recognized_net_revenue,cancelled_orders,returned_orders
0,2026-01-01,14,2659.87,-189.99,2469.88,1.0,1.0
1,2026-02-01,16,2874.85,-124.99,2749.86,1.0,1.0
2,2026-03-01,17,2739.85,-79.99,2659.86,2.0,1.0
3,2026-04-01,13,3069.87,-399.99,2669.88,0.0,1.0


In [6]:
fig = px.line(
    finance,
    x='revenue_month',
    y='recognized_net_revenue',
    markers=True,
    title='Recognized Net Revenue by Payment Month',
    labels={'revenue_month': 'Month', 'recognized_net_revenue': 'Recognized Net Revenue'},
)
fig.show()

#### GitHub-renderable static fallback

This static image is exported from the Plotly chart for GitHub preview; the local notebook version remains interactive.

![Recognized net revenue by payment month](../assets/charts/finance-recognized-net-revenue.svg)


## Marketing View

Marketing uses campaign-attributed net revenue after cancellations and returns are handled.

In [7]:
marketing

,marketing_source,is_promo_attributed,orders,gross_revenue,returns_deducted,net_revenue,pct_net_revenue
0,direct,False,30,6784.70,-124.99,6659.71,63.1
1,email,True,11,1479.90,0.00,1479.90,14.0
2,organic,False,12,1409.89,-269.98,1139.91,10.8
3,affiliate,True,3,789.98,0.00,789.98,7.5
4,paid_search,True,4,879.97,-399.99,479.98,4.5


In [8]:
fig = px.bar(
    marketing,
    x='marketing_source',
    y='net_revenue',
    color='is_promo_attributed',
    title='Net Revenue by Marketing Source',
    labels={'marketing_source': 'Marketing Source', 'net_revenue': 'Net Revenue'},
)
fig.show()

#### GitHub-renderable static fallback

This static image is exported from the Plotly chart for GitHub preview; the local notebook version remains interactive.

![Net revenue by marketing source](../assets/charts/marketing-net-revenue-by-source.svg)


## Ecommerce View

Ecommerce uses channel and product-category performance to understand demand mix, AOV, and return rate.

In [9]:
channel.head(12)

,channel_name,channel_group,product_category,orders,gross_revenue,returns_deducted,net_revenue,average_order_value,return_rate_pct
0,Retail,Partner,Equipment,5,2089.95,0.00,2089.95,417.99,0.0
1,Wholesale,Partner,Equipment,3,1329.97,0.00,1329.97,443.32,0.0
2,B2B,Partner,Footwear,5,989.95,0.00,989.95,197.99,0.0
3,B2B,Partner,Apparel,7,979.93,0.00,979.93,139.99,0.0
4,B2B,Partner,Equipment,1,899.99,0.00,899.99,899.99,0.0
5,Direct Web,Owned,Equipment,3,1189.97,-399.99,789.98,396.66,33.3
6,Direct Web,Owned,Footwear,6,969.95,-189.99,779.96,161.66,16.7
7,Amazon,Partner,Equipment,3,559.98,0.00,559.98,186.66,0.0
8,Direct Web,Owned,Apparel,5,519.96,0.00,519.96,103.99,0.0
9,Retail,Partner,Footwear,3,489.97,0.00,489.97,163.32,0.0


In [10]:
fig = px.bar(
    channel.head(12),
    x='channel_name',
    y='net_revenue',
    color='product_category',
    title='Net Revenue by Channel and Product Category',
    labels={'channel_name': 'Channel', 'net_revenue': 'Net Revenue'},
)
fig.show()

#### GitHub-renderable static fallback

This static image is exported from the Plotly chart for GitHub preview; the local notebook version remains interactive.

![Net revenue by channel and product category](../assets/charts/ecommerce-channel-category-performance.svg)


## Data Quality View

The data quality mart shows which raw issues create reconciliation risk.

In [11]:
quality

,issue_type,affected_rows,affected_orders,raw_gross_revenue_at_risk
0,cancelled_revenue_zeroed,4,4,809.96
1,date_timing_difference,4,4,1839.96
2,return_standardized,4,4,794.96
3,duplicate,3,3,729.97
4,duplicate_removed,3,3,729.97
5,pipeline_excluded,3,3,259.98


In [12]:
fig = px.bar(
    quality,
    x='issue_type',
    y='affected_rows',
    title='Data Quality Issues Found in Raw Source',
    labels={'issue_type': 'Issue Type', 'affected_rows': 'Affected Rows'},
)
fig.update_layout(xaxis_tickangle=-25)
fig.show()

#### GitHub-renderable static fallback

This static image is exported from the Plotly chart for GitHub preview; the local notebook version remains interactive.

![Data quality issues found in raw source](../assets/charts/data-quality-issues.svg)


## Recommendation

Use `fct_orders` as the governed revenue backbone. Finance, Marketing, and Ecommerce should all consume downstream marts from that same fact table so the differences between metrics are intentional, documented, and reconcilable.